# Fink LSST — SALT2 Fit of `sn_near_galaxy_candidate` Light Curves using Photometric Redshift

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-05-11
- **last update**: 2026-05-12 
- **based on** :
  - `02_fink_sn_near_galaxy_candidate_sn_lightcurves.ipynb` (catalog & LC download for `sn_near_galaxy_candidate`)
  - `03_fink_tns_sn_fitSNIa_salt2_lightcurves.ipynb` (SALT2 fit with fixed TNS spec-z)

## Purpose

This notebook selects **`sn_near_galaxy_candidate`** alerts from the Fink broker
that have a **photometric redshift** available from the near host galaxy
(field `f:xm_legacydr8_zphot` from Legacy Survey DR8, or
`f:xm_mangrove_lum_dist` converted to a redshift as fallback).

For each such event, the SALT2 model is fitted with the **photometric redshift
held fixed** — unlike notebook 04 where z is free, here we trust the host
galaxy photo-z as a prior.

The output is a **Hubble diagram** μ(z_phot) compared to the ΛCDM prediction,
where:
- **Colour** encodes the quality of the SALT2 fit (χ²/dof).
- **Marker shape** encodes the galaxy type (from `f:xm_legacydr8_fqual`,
  `f:xm_simbad_otype`, or MANGROVE HyperLEDA morphology when available).

## Data flow
```
Fink LSST API  →  tag: sn_near_galaxy_candidate
  ↓  (this notebook downloads and caches everything)
  ├─ data_NB07_05_SN_NEAR_SALT2/catalog_sn_near_galaxy_zphot.parquet
  ├─ data_NB07_05_SN_NEAR_SALT2/light_curves/lc_<diaObjectId>.parquet
  ├─ data_NB07_05_SN_NEAR_SALT2/salt2_fit_results.parquet
  └─ figs_NB07_05_SN_NEAR_SALT2/
```

## Column conventions
- `r:<col>` — LSST diaSource/diaObject field (prefix = table name, NOT r-band)
- `f:<col>` — Fink-computed field
- Spectral band → `r:band` ∈ {u, g, r, i, z, y}
- Flux unit: **nJy** (nano-Jansky, AB system, ZP = 31.4)

## Redshift priority
1. `f:xm_legacydr8_zphot` — photometric z from Legacy Survey DR8 (preferred)
2. `f:xm_tns_redshift` — spectroscopic z from TNS (used if DR8 photo-z absent)

## References
- sncosmo: https://sncosmo.readthedocs.io
- Guy et al. 2007 (SALT2); Betoule et al. 2014 (Tripp formula)
- Dey et al. 2019 (Legacy Survey DR8)


## 0 — Imports

In [ ]:
import io
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import astropy.table
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import requests
import sncosmo
from scipy.integrate import quad

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Imports OK")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → %matplotlib inline")

## 1 — Configuration

In [ ]:
# ── Fink LSST API ─────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org/api/v1"
TAG = "sn_near_galaxy_candidate"
N_ALERTS = 5000
API_DELAY = 0.4  # seconds between API calls

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "NB07_05_FINK_SN_NEARGAL_FIT_SALT2"
DATA_DIR = Path(f"data_{NB_TAG}")
FIGS_DIR = Path(f"figs_{NB_TAG}")
LC_DIR = DATA_DIR / "light_curves"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)
LC_DIR.mkdir(parents=True, exist_ok=True)

# ── Photometric system ────────────────────────────────────────────────────────
RUBIN_ZP = 31.4  # AB magnitude zero-point for nJy
ZPSYS = "ab"

# ── SALT2 model ───────────────────────────────────────────────────────────────
SALT2_SOURCE = "salt2-extended"
BAND_PREFIX = "lsst"  # sncosmo band names: lsstu, lsstg, lsstr, ...
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]

# ── Object selection ──────────────────────────────────────────────────────────
Z_MIN = 0.01  # minimum photometric redshift accepted for the fit
Z_MAX = 3.0  # maximum photometric redshift accepted for the fit
SNR_MIN = 3.0  # minimum per-point SNR (psfFlux / psfFluxErr)
MIN_DATAPOINTS = 5  # minimum number of SNR-passing data points required

# ── Redshift column priority ──────────────────────────────────────────────────
# Column 1 (preferred): photo-z from Legacy Survey DR8 cross-match
ZPHOT_COL = "f:xm_legacydr8_zphot"
ZPHOT_ERR_COL = "f:xm_legacydr8_e_zphot"
# Column 2 (fallback): spectroscopic z from TNS
ZSPEC_COL = "f:xm_tns_redshift"

# ── Tripp formula nuisance parameters (Betoule+ 2014) ────────────────────────
ALPHA = 0.14
BETA = 3.14
M_B = -19.3

# ── ΛCDM cosmology ────────────────────────────────────────────────────────────
H0 = 70.0
OmegaM = 0.30
OmegaL = 0.70

# ── Band colours ──────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#3498DB",  # blue
    "g": "#27AE60",  # green
    "r": "#E74C3C",  # red
    "i": "#F1C40F",  # yellow
    "z": "#8B4513",  # brown
    "y": "#95A5A6",  # grey
}

# ── Dark theme ────────────────────────────────────────────────────────────────
DARK_BG = "#0D1117"
PANEL_BG = "#161B22"
TEXT_COL = "#E6EDF3"
MUTED_COL = "#8B949E"
ACCENT = "#58A6FF"

# ── Hubble diagram quality cuts ───────────────────────────────────────────────
CHI2_RED_MAX = 10.0  # maximum reduced chi2 to be included in Hubble diagram
X1_MAX = 6.0  # maximum |x1|
C_MAX = 1.0  # maximum |c|

# ── NCOLS for multi-panel LC plot ─────────────────────────────────────────────
NCOLS_LC = 3

print(f"Tag          : {TAG}")
print(f"N alerts     : {N_ALERTS}")
print(f"SALT2 source : {SALT2_SOURCE}")
print(f"ZP={RUBIN_ZP}  zpsys={ZPSYS}")
print(f"Tripp α={ALPHA}  β={BETA}  M_B={M_B}")
print(f"ΛCDM H0={H0}  Ωm={OmegaM}  ΩΛ={OmegaL}")
print(f"Redshift (primary)  : {ZPHOT_COL}")
print(f"Redshift (fallback) : {ZSPEC_COL}")

## 2 — Download catalog from Fink API

We query `/api/v1/tags` with `tag=sn_near_galaxy_candidate`.
We request all columns relevant to the near-galaxy photo-z, host galaxy type,
TNS classification (for comparison only), and Fink SN classifiers.

In [ ]:
CATALOG_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:band",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:snr",
        "r:reliability",
        # Fink classifiers
        "f:clf_cats_class",
        "f:clf_cats_score",
        "f:clf_snnSnVsOthers_score",
        "f:clf_earlySNIa_score",
        # TNS cross-match (used for comparison / validation only)
        "f:xm_tns_fullname",
        "f:xm_tns_type",
        "f:xm_tns_redshift",
        # Legacy Survey DR8 (near galaxy photo-z — primary redshift source)
        "f:xm_legacydr8_zphot",
        "f:xm_legacydr8_e_zphot",
        "f:xm_legacydr8_fqual",
        "f:xm_legacydr8_pstar",
        # SIMBAD object type (galaxy morphology proxy)
        "f:xm_simbad_otype",
        # MANGROVE galaxy catalogue (luminosity distance, morphology)
        "f:xm_mangrove_lum_dist",
        "f:xm_mangrove_2MASS_name",
        "f:xm_mangrove_HyperLEDA_name",
        "f:xm_mangrove_ang_dist",
        # Gaia DR3
        "f:xm_gaiadr3_DR3Name",
        "f:xm_gaiadr3_VarFlag",
    ]
)


def fetch_catalog(tag: str, n: int) -> pd.DataFrame:
    """Query /api/v1/tags endpoint for a given Fink LSST tag."""
    params = {
        "tag": tag,
        "n": n,
        "columns": CATALOG_COLUMNS,
        "output-format": "json",
    }
    print(f"Querying {FINK_API}/tags  tag={tag}  n={n} ...")
    resp = requests.get(f"{FINK_API}/tags", params=params, timeout=120)
    resp.raise_for_status()
    if not resp.text.strip():
        print("[warn] Empty response from API.")
        return pd.DataFrame()
    df = pd.read_json(io.BytesIO(resp.content))
    print(f"  -> {len(df)} alerts received.")
    return df


raw_catalog = fetch_catalog(TAG, N_ALERTS)

In [ ]:
# Deduplicate: one row per diaObjectId, keeping the most recent alert
if "r:midpointMjdTai" in raw_catalog.columns:
    catalog = (
        raw_catalog.sort_values("r:midpointMjdTai")
        .drop_duplicates(subset="r:diaObjectId", keep="last")
        .reset_index(drop=True)
    )
else:
    catalog = raw_catalog.drop_duplicates(subset="r:diaObjectId").reset_index(drop=True)

print(f"Unique diaObjects after deduplication: {len(catalog)}")
catalog.head(5)

In [ ]:
# Save full catalog
cat_path = DATA_DIR / "catalog_sn_near_galaxy_full.parquet"
catalog.to_parquet(cat_path, index=False)
print(f"Full catalog saved -> {cat_path}")

## 3 — Assign photometric redshift and galaxy type

### 3.1 Redshift priority
- Primary: `f:xm_legacydr8_zphot`
- Fallback: `f:xm_tns_redshift` (spectroscopic from TNS)

### 3.2 Galaxy type classification
We derive a simplified galaxy-type label for each object from the available
cross-match metadata:
- `f:xm_simbad_otype` — SIMBAD object type (contains 'G', 'Gx', 'LSB', etc.)
- `f:xm_legacydr8_pstar` — Legacy DR8 star/galaxy separator (>0.5 → likely star)
- `f:xm_legacydr8_fqual` — Legacy DR8 fit quality flag

The galaxy type labels are:
- `elliptical` / `spiral` / `irregular` — from SIMBAD otype keywords
- `galaxy_DR8` — host detected in Legacy DR8 but morphology unknown
- `mangrove` — host only in MANGROVE catalogue
- `unknown` — no host galaxy information


In [ ]:
def assign_redshift(row: pd.Series) -> tuple[float, str]:
    """Return (z_best, z_source) following the redshift priority.

    Returns
    -------
    z_best   : best redshift estimate (float, NaN if unavailable)
    z_source : one of 'legacydr8_phot', 'tns_spec', or 'none'
    """
    try:
        z_phot = float(row.get(ZPHOT_COL, np.nan))
    except (TypeError, ValueError):
        z_phot = np.nan

    try:
        z_spec = float(row.get(ZSPEC_COL, np.nan))
    except (TypeError, ValueError):
        z_spec = np.nan

    if np.isfinite(z_phot) and Z_MIN <= z_phot <= Z_MAX:
        return z_phot, "legacydr8_phot"
    elif np.isfinite(z_spec) and Z_MIN <= z_spec <= Z_MAX:
        return z_spec, "tns_spec"
    return np.nan, "none"


# SIMBAD otype keywords for morphological classification
_ELLIPTICAL_KEYS = ["EG", "E+", "lEG", "BCG", "S0", "elliptical", "Ell"]
_SPIRAL_KEYS = ["SBG", "SAG", "SyG", "spiral", "SAB", "SB", "Sp", "barred"]
_IRREGULAR_KEYS = ["LSB", "Irr", "LMC", "dSph", "dE", "BCD", "HII", "irregular"]


def classify_galaxy_type(row: pd.Series) -> str:
    """Assign a simplified galaxy morphology label from available cross-match metadata.

    Priority: SIMBAD otype keywords > Legacy DR8 detection > MANGROVE > unknown
    """
    simbad = str(row.get("f:xm_simbad_otype", "") or "")

    # Check SIMBAD keywords
    if any(k in simbad for k in _ELLIPTICAL_KEYS):
        return "elliptical"
    if any(k in simbad for k in _SPIRAL_KEYS):
        return "spiral"
    if any(k in simbad for k in _IRREGULAR_KEYS):
        return "irregular"
    if "G" in simbad or "Gx" in simbad or "Galaxy" in simbad or simbad.startswith("G"):
        return "galaxy_simbad"

    # Legacy DR8 detection
    try:
        z_phot = float(row.get(ZPHOT_COL, np.nan))
        if np.isfinite(z_phot):
            pstar = float(row.get("f:xm_legacydr8_pstar", 1.0) or 1.0)
            return "galaxy_DR8" if pstar < 0.5 else "star_DR8"
    except (TypeError, ValueError):
        pass

    # MANGROVE detection
    mangrove_name = str(row.get("f:xm_mangrove_HyperLEDA_name", "") or "")
    if mangrove_name and mangrove_name not in ("", "None", "nan"):
        return "mangrove"

    return "unknown"


# Apply to full catalog
z_info = catalog.apply(assign_redshift, axis=1)
catalog["z_best"] = [zi[0] for zi in z_info]
catalog["z_source"] = [zi[1] for zi in z_info]
catalog["galaxy_type"] = catalog.apply(classify_galaxy_type, axis=1)

print(f"Redshift availability:")
print(catalog["z_source"].value_counts().to_string())
print(f"\nGalaxy type distribution:")
print(catalog["galaxy_type"].value_counts().to_string())

### 3.2 Select objects with a usable redshift

In [ ]:
# Keep only objects for which we have a valid photo-z (or fallback spec-z)
sel_z = catalog["z_source"] != "none"
zphot_catalog = catalog[sel_z].copy().reset_index(drop=True)

print(f"Objects with usable redshift: {len(zphot_catalog)} / {len(catalog)}")
print(f"  z_source breakdown: {zphot_catalog['z_source'].value_counts().to_dict()}")
print(f"  z_best range: [{zphot_catalog['z_best'].min():.3f}, {zphot_catalog['z_best'].max():.3f}]")

# Save filtered catalog
zphot_catalog.to_parquet(DATA_DIR / "catalog_sn_near_galaxy_zphot.parquet", index=False)
print(f"Filtered catalog saved -> {DATA_DIR}/catalog_sn_near_galaxy_zphot.parquet")

### 3.3 Overview plots: redshift and galaxy type distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), facecolor=DARK_BG)

# --- Redshift distribution ---
ax = axes[0]
ax.set_facecolor(PANEL_BG)
z_phot_vals = pd.to_numeric(
    zphot_catalog.loc[zphot_catalog["z_source"] == "legacydr8_phot", "z_best"], errors="coerce"
).dropna()
z_spec_vals = pd.to_numeric(
    zphot_catalog.loc[zphot_catalog["z_source"] == "tns_spec", "z_best"], errors="coerce"
).dropna()
if len(z_phot_vals) > 0:
    ax.hist(
        z_phot_vals, bins=30, color="#58A6FF", alpha=0.8, label=f"LegacyDR8 phot-z (N={len(z_phot_vals)})"
    )
if len(z_spec_vals) > 0:
    ax.hist(z_spec_vals, bins=30, color="#FF7B72", alpha=0.8, label=f"TNS spec-z (N={len(z_spec_vals)})")
ax.set_xlabel("Redshift", color=MUTED_COL)
ax.set_ylabel("Count", color=MUTED_COL)
ax.set_title("Redshift distribution", color=ACCENT)
ax.tick_params(colors=MUTED_COL)
ax.legend(fontsize=8, facecolor=DARK_BG, labelcolor=TEXT_COL)

# --- Galaxy type bar chart ---
ax = axes[1]
ax.set_facecolor(PANEL_BG)
gtype_counts = zphot_catalog["galaxy_type"].value_counts()
colors_bar = ["#58A6FF", "#FF7B72", "#3FB950", "#D2A8FF", "#F78166", "#79C0FF"]
bars = ax.barh(
    gtype_counts.index[::-1],
    gtype_counts.values[::-1],
    color=colors_bar[: len(gtype_counts)],
    edgecolor=DARK_BG,
)
ax.set_xlabel("Number of objects", color=MUTED_COL)
ax.set_title("Galaxy type distribution", color=ACCENT)
ax.tick_params(colors=MUTED_COL, labelsize=8)

# --- Sky map ---
ax = axes[2]
ax.set_facecolor(PANEL_BG)
sc = ax.scatter(
    zphot_catalog["r:ra"],
    zphot_catalog["r:dec"],
    c=zphot_catalog["z_best"],
    cmap="plasma",
    s=15,
    alpha=0.8,
    vmin=0,
    vmax=min(1.0, zphot_catalog["z_best"].quantile(0.95)),
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("z_best", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.get_yticklabels(), color="white")
ax.set_xlabel("RA (deg)", color=MUTED_COL)
ax.set_ylabel("Dec (deg)", color=MUTED_COL)
ax.set_title("Sky map (colour = z_best)", color=ACCENT)
ax.tick_params(colors=MUTED_COL)

plt.suptitle(f"Fink LSST — {TAG} with redshift — N={len(zphot_catalog)}", color=TEXT_COL, fontsize=12, y=1.01)
fig.tight_layout()
fig.savefig(FIGS_DIR / "catalog_overview.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()

## 4 — Download light curves from Fink API

In [ ]:
LC_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:band",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:snr",
        "r:reliability",
    ]
)


def fetch_lightcurve(obj_id: int, cache_dir: Path = LC_DIR) -> pd.DataFrame:
    """Download (or load from cache) the full alert history for one diaObjectId."""
    cache_path = cache_dir / f"lc_{obj_id}.parquet"
    if cache_path.exists():
        return pd.read_parquet(cache_path)

    params = {
        "diaObjectId": obj_id,
        "columns": LC_COLUMNS,
        "output-format": "json",
    }
    try:
        resp = requests.get(f"{FINK_API}/sources", params=params, timeout=60)
        resp.raise_for_status()
        if not resp.text.strip():
            return pd.DataFrame()
        df = pd.read_json(io.BytesIO(resp.content))
        if "r:midpointMjdTai" in df.columns:
            df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)
        df.to_parquet(cache_path, index=False)
        time.sleep(API_DELAY)
        return df
    except Exception as exc:
        print(f"  [error] diaObjectId={obj_id}: {exc}")
        return pd.DataFrame()


print(f"fetch_lightcurve ready. Cache dir: {LC_DIR}")

In [ ]:
lc_cache: dict[int, pd.DataFrame] = {}
n_ok, n_fail = 0, 0

for i, row in zphot_catalog.iterrows():
    oid = int(row["r:diaObjectId"])
    lc = fetch_lightcurve(oid)
    lc_cache[oid] = lc
    if not lc.empty:
        n_ok += 1
    else:
        n_fail += 1
    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(zphot_catalog)} done  (ok={n_ok}, fail={n_fail})")

print(f"\nLight curve download complete: ok={n_ok}, fail={n_fail}")

## 5 — Build sncosmo observation tables and run SALT2 fits

The photometric redshift `z_best` is held **fixed** during the fit.
Free parameters: `t0`, `x0`, `x1`, `c`.

In [ ]:
def make_sncosmo_table(lc_df: pd.DataFrame) -> astropy.table.Table:
    """Convert a Fink LSST alert DataFrame to an astropy.Table for sncosmo.

    Applies quality cuts: fluxerr > 0, SNR >= SNR_MIN.
    Band names are converted to the sncosmo convention: 'u' → 'lsstu', etc.
    """
    df = lc_df.rename(
        columns={
            "r:midpointMjdTai": "time",
            "r:band": "band_raw",
            "r:psfFlux": "flux",
            "r:psfFluxErr": "fluxerr",
            "r:snr": "snr",
        }
    ).copy()

    df = df[df["fluxerr"] > 0].copy()
    if "snr" in df.columns:
        df = df[df["snr"] >= SNR_MIN].copy()

    df["band"] = BAND_PREFIX + df["band_raw"].str.strip().str.lower()

    return astropy.table.Table(
        {
            "time": df["time"].values.astype(float),
            "band": df["band"].values,
            "flux": df["flux"].values.astype(float),
            "fluxerr": df["fluxerr"].values.astype(float),
            "zp": np.full(len(df), RUBIN_ZP, dtype=float),
            "zpsys": np.full(len(df), ZPSYS),
        }
    )


def fit_salt2_fixed_z(lc_df: pd.DataFrame, z_fixed: float) -> dict:
    """Fit a SALT2 model on a Fink LSST light curve with z held fixed.

    Parameters
    ----------
    lc_df   : raw Fink alert DataFrame for one diaObjectId
    z_fixed : redshift to hold fixed (photo-z from host galaxy)

    Returns
    -------
    dict with keys: success, z, t0, x0, x1, c, chi2, ndof, chi2_red, mB,
                    mu_tripp, result, fitted_model, table, message
    """
    try:
        obs = make_sncosmo_table(lc_df)
    except Exception as exc:
        return {"success": False, "message": f"Table build failed: {exc}"}

    if len(obs) < MIN_DATAPOINTS:
        return {
            "success": False,
            "message": f"Only {len(obs)} points after quality cuts (need {MIN_DATAPOINTS})",
            "table": obs,
        }

    model = sncosmo.Model(source=SALT2_SOURCE)

    # Initial parameter guesses
    peak_idx = int(np.argmax(obs["flux"]))
    t0_guess = float(obs["time"][peak_idx])
    model.set(z=z_fixed, t0=t0_guess, x0=1e-3, x1=0.0, c=0.0)

    vparam_names = ["t0", "x0", "x1", "c"]
    bounds = {
        "t0": (t0_guess - 50.0, t0_guess + 50.0),
        "x0": (1e-10, 1e2),
        "x1": (-5.0, 5.0),
        "c": (-0.5, 0.5),
    }

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result, fitted_model = sncosmo.fit_lc(
                obs,
                model,
                vparam_names=vparam_names,
                bounds=bounds,
                minsnr=0.0,
                warn=False,
            )
    except Exception as exc:
        return {"success": False, "message": f"Fit failed: {exc}", "table": obs}

    chi2 = result.chisq
    ndof = result.ndof
    chi2_red = chi2 / max(ndof, 1)

    # Tripp distance modulus
    SALT2_ZP_INTERNAL = 10.635
    x0_fit = float(fitted_model["x0"])
    x1_fit = float(fitted_model["x1"])
    c_fit = float(fitted_model["c"])
    mB = -2.5 * np.log10(max(x0_fit, 1e-30)) + SALT2_ZP_INTERNAL
    mu = mB - M_B + ALPHA * x1_fit - BETA * c_fit

    return {
        "success": True,
        "result": result,
        "fitted_model": fitted_model,
        "table": obs,
        "z": z_fixed,
        "t0": float(fitted_model["t0"]),
        "x0": x0_fit,
        "x1": x1_fit,
        "c": c_fit,
        "chi2": chi2,
        "ndof": ndof,
        "chi2_red": chi2_red,
        "mB": mB,
        "mu_tripp": mu,
        "message": result.message,
    }


print("SALT2 fit helpers ready.")

In [ ]:
def computemustaterrors(
    param_names, params, covariance, mb_const=10.635, M=-19.3, alpha=0.14, beta=3.1, cosmo=None
):
    """Propagate SALT2 fit covariance into an uncertainty on the Tripp distance modulus.

    Tripp formula
    -------------
        mu  =  m_B*  -  M  +  alpha*x1  -  beta*c
        m_B*  =  -2.5 * log10(x0)  +  mb_const

    Partial derivatives of mu (gradient vector g):
        d(mu)/d(x0) = -2.5 / (x0 * ln10)
        d(mu)/d(x1) =  alpha
        d(mu)/d(c)  = -beta
        d(mu)/d(z)  = (5/ln10)*(1/d_L)*dd_L/dz   [only when z is a free parameter]

    The full error propagation is evaluated as:
        sigma_mu = sqrt( g^T . C_sub . g )
    where C_sub is the covariance sub-matrix restricted to the parameters
    that enter mu (x0, x1, c and optionally z).  t0 is excluded.

    Parameters
    ----------
    param_names : list[str]
        Names of *all* fitted parameters, in the same order as `params` and
        the rows/columns of `covariance`.
        z fixed : ['t0', 'x0', 'x1', 'c']
        z free  : ['z', 't0', 'x0', 'x1', 'c']
    params      : array-like  – fitted values, same order as param_names.
    covariance  : 2-D array   – full covariance matrix, same order.
    mb_const    : SALT2 zero-point offset (default 10.635 for salt2-extended).
    M           : absolute B-band magnitude of a standard SNIa.
    alpha       : Tripp stretch coefficient.
    beta        : Tripp colour coefficient.
    cosmo       : astropy cosmology instance; used only when z is free.
                  Defaults to FlatLambdaCDM(H0=70, Om0=0.3).

    Returns
    -------
    dict with keys:
        mu          – Tripp distance modulus
        sigma_mu    – 1-sigma uncertainty on mu
        gradient    – {param_name: d(mu)/d(param)} for mu-entering params
        sigmas      – {param_name: 1-sigma marginal error} for all params
        covariances – {'pi:pj': cov(pi,pj)} for mu-entering pairs
    """
    params = np.asarray(params, dtype=float)
    cov = np.asarray(covariance, dtype=float)
    names = list(param_names)

    # Build a name→index map so we never hard-code positions
    idx = {name: i for i, name in enumerate(names)}

    # Retrieve the three SALT2 photometric parameters by name
    x0 = params[idx["x0"]]
    x1 = params[idx["x1"]]
    c = params[idx["c"]]

    # Tripp distance modulus
    mB_star = -2.5 * np.log10(x0) + mb_const
    mu = mB_star - M + alpha * x1 - beta * c

    # Gradient of mu w.r.t. the parameters that enter it
    gradient = {
        "x0": -2.5 / (x0 * np.log(10)),  # d(m_B*)/d(x0)
        "x1": alpha,  # d(mu)/d(x1)
        "c": -beta,  # d(mu)/d(c)
    }
    mu_params = ["x0", "x1", "c"]

    # Add the z contribution only when z was a free parameter
    if "z" in idx:
        z = params[idx["z"]]
        if cosmo is None:
            from astropy.cosmology import FlatLambdaCDM

            cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
        dz = 1e-5
        dL = cosmo.luminosity_distance(z).value
        dL_p = cosmo.luminosity_distance(z + dz).value
        dL_m = cosmo.luminosity_distance(z - dz).value
        gradient["z"] = (5.0 / np.log(10)) * (dL_p - dL_m) / (2.0 * dz * dL)
        mu_params.append("z")

    # Gradient vector g and covariance sub-matrix C_sub
    # restricted to the parameters that actually enter mu  (t0 excluded)
    g = np.array([gradient[p] for p in mu_params])
    sub_idx = np.array([idx[p] for p in mu_params])
    C_sub = cov[np.ix_(sub_idx, sub_idx)]

    # sigma_mu^2 = g^T . C_sub . g  (exact first-order error propagation)
    sigma_mu = float(np.sqrt(max(0.0, g @ C_sub @ g)))

    # Marginal 1-sigma errors on every fitted parameter
    sigmas = {name: float(np.sqrt(max(0.0, cov[i, i]))) for name, i in idx.items()}

    # Off-diagonal covariances for mu-entering pairs (upper triangle only)
    covariances = {}
    for ia, a in enumerate(mu_params):
        for ib, b in enumerate(mu_params):
            if ia < ib:
                covariances[f"{a}:{b}"] = float(C_sub[ia, ib])

    return {
        "mu": float(mu),
        "sigma_mu": sigma_mu,
        "gradient": gradient,
        "sigmas": sigmas,
        "covariances": covariances,
    }


print("computemustaterrors ready  (gradient g^T.C.g, param_names-aware)")

In [ ]:
# Verify LSST band registration in sncosmo
print("Checking LSST band registration in sncosmo:")
for b in BAND_ORDER:
    bname = BAND_PREFIX + b
    try:
        bp = sncosmo.get_bandpass(bname)
        print(f"  {bname:10s}  ✓")
    except Exception as exc:
        print(f"  {bname:10s}  ✗  {exc}")

In [ ]:
# Run SALT2 fits for all objects with a photometric redshift
fit_results: dict[int, dict] = {}
fit_errors: dict[int, dict] = {}

for idx, row in zphot_catalog.iterrows():
    oid = int(row["r:diaObjectId"])
    z_best = float(row["z_best"])
    z_src = row["z_source"]
    gtype = row["galaxy_type"]
    tns_name = row.get("f:xm_tns_fullname", str(oid))

    lc_df = lc_cache.get(oid, pd.DataFrame())
    if lc_df.empty:
        fit_results[oid] = {"success": False, "message": "No light curve data"}
        print(f"  [{str(oid):20s}]  ✗  No LC data")
        continue

    # Do the fit here
    res = fit_salt2_fixed_z(lc_df, z_fixed=z_best)
    fit_results[oid] = res

    if res["success"] and res["result"].covariance is not None:
        # retrieve errors
        param_names = res["result"].vparam_names
        # print( "param_names = ",param_names,"   params = ",params,"  cov = ",cov)
        params = res["result"].parameters
        cov = res["result"].covariance
        fit_errors[oid] = computemustaterrors(param_names, params, cov)
        sigma_mu = fit_errors[oid]["sigma_mu"]

        print(
            f"  [{str(tns_name)[:20]:20s}]  z={z_best:.3f} ({z_src[:4]})  "
            f"x1={res['x1']:+.3f}  c={res['c']:+.3f}  "
            f"χ²/dof={res['chi2_red']:.2f}  μ={res['mu_tripp']:.2f}  [{gtype}]"
            f"σ_mu={sigma_mu:.3f}"
        )
    else:
        print(f"  [{str(tns_name)[:20]:20s}]  z={z_best:.3f}  ✗  {res['message']}")
        fit_results[oid]["success"] = False
        fit_errors[oid] = None


n_ok = sum(1 for r in fit_results.values() if r["success"])
print(f"\nSummary: {n_ok}/{len(fit_results)} fits converged.")

## 6 — Collect results in a DataFrame

In [ ]:
rows = []
for oid, res in fit_results.items():
    if not res["success"]:
        continue
    meta = zphot_catalog.loc[zphot_catalog["r:diaObjectId"] == oid].iloc[0]
    #    sigma_mu = fit_errors[oid]["sigma_mu"]
    rows.append(
        {
            "diaObjectId": oid,
            "tns_name": meta.get("f:xm_tns_fullname", ""),
            "tns_type": meta.get("f:xm_tns_type", ""),
            "galaxy_type": meta["galaxy_type"],
            "z_best": float(meta["z_best"]),
            "z_source": meta["z_source"],
            "z_phot_err": float(meta.get(ZPHOT_ERR_COL, np.nan) or np.nan),
            "RA": float(meta["r:ra"]),
            "Dec": float(meta["r:dec"]),
            "t0": res["t0"],
            "x0": res["x0"],
            "x1": res["x1"],
            "c": res["c"],
            "mB": res["mB"],
            "mu_tripp": res["mu_tripp"],
            "chi2": res["chi2"],
            "ndof": res["ndof"],
            "chi2_red": res["chi2_red"],
            #            "sigma_mu":sigma_mu,
            "SNN_score": float(meta.get("f:clf_snnSnVsOthers_score", np.nan) or np.nan),
            "earlySNIa_score": float(meta.get("f:clf_earlySNIa_score", np.nan) or np.nan),
        }
    )

results_df = pd.DataFrame(rows).sort_values("z_best").reset_index(drop=True)
print(f"{len(results_df)} successful SALT2 fits.")

results_df.to_parquet(DATA_DIR / "salt2_fit_results.parquet", index=False)
results_df.to_csv(DATA_DIR / "salt2_fit_results.csv", index=False)
print(f"Results saved -> {DATA_DIR}/salt2_fit_results.*")
results_df

## 7 — Multi-panel SALT2 fit plots

In [ ]:
def plot_salt2_fit(ax, oid: int, meta: pd.Series, res: dict):
    """Plot raw Fink data points + SALT2 model curves for one object in a given Axes."""
    if not res.get("success"):
        ax.text(
            0.5, 0.5, "FIT FAILED", transform=ax.transAxes, ha="center", va="center", color="red", fontsize=11
        )
        ax.set_title(
            f"{meta.get('f:xm_tns_fullname', oid)}\n{res.get('message', '')}", fontsize=7, color="red"
        )
        return

    obs = res["table"]
    fitted_model = res["fitted_model"]
    t0 = res["t0"]
    t_dense = np.linspace(float(obs["time"].min()) - 15, float(obs["time"].max()) + 15, 500)

    for b in BAND_ORDER:
        bname = BAND_PREFIX + b
        color = BAND_COLORS.get(b, "gray")
        mask = np.array(obs["band"]) == bname
        if mask.sum() > 0:
            ax.errorbar(
                obs["time"][mask] - t0,
                obs["flux"][mask],
                yerr=obs["fluxerr"][mask],
                color=color,
                fmt="o",
                markersize=4,
                capsize=2,
                lw=0.8,
                label=b,
                zorder=3,
            )
        try:
            f_model = fitted_model.bandflux(bname, t_dense, zp=RUBIN_ZP, zpsys=ZPSYS)
            valid = np.isfinite(f_model)
            if valid.sum() > 1:
                ax.plot(t_dense[valid] - t0, f_model[valid], color=color, lw=1.5, zorder=2)
        except Exception:
            pass

    ax.axhline(0.0, color="k", lw=0.5, ls="--")
    name = meta.get("f:xm_tns_fullname", str(oid))
    z_src = meta["z_source"]
    gtype = meta["galaxy_type"]
    ax.set_title(
        f"{name}  z={res['z']:.3f} ({z_src[:4]})\n"
        f"x1={res['x1']:+.2f}  c={res['c']:+.2f}  χ²/dof={res['chi2_red']:.2f}  [{gtype}]",
        fontsize=7,
    )
    ax.set_xlabel(r"$t - t_0$ [days]", fontsize=8)
    ax.set_ylabel("psfFlux [nJy]", fontsize=8)
    ax.tick_params(axis="both", labelsize=7)
    ax.legend(fontsize=6, ncol=3, loc="upper right")

In [ ]:
# Build list of objects to plot (only successful fits)
ok_list = [
    (oid, zphot_catalog.loc[zphot_catalog["r:diaObjectId"] == oid].iloc[0])
    for oid in fit_results
    if fit_results[oid]["success"]
]

n_plots = len(ok_list)
if n_plots == 0:
    print("No successful fits to plot.")
else:
    nrows = math.ceil(n_plots / NCOLS_LC)
    fig, axes = plt.subplots(nrows, NCOLS_LC, figsize=(6.5 * NCOLS_LC, 4.5 * nrows), squeeze=False)
    for k, (oid, row) in enumerate(ok_list):
        ax = axes[k // NCOLS_LC][k % NCOLS_LC]
        plot_salt2_fit(ax, oid, row, fit_results[oid])

    for k in range(n_plots, nrows * NCOLS_LC):
        axes[k // NCOLS_LC][k % NCOLS_LC].set_visible(False)

    fig.suptitle(
        f"SALT2 fits — Fink/LSST  tag={TAG}  (z fixed to host photo-z)",
        fontsize=12,
        y=1.002,
    )
    fig.tight_layout()
    fig_path = FIGS_DIR / "salt2_fits_lightcurves.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Figure saved → {fig_path}")
    plt.show()

## 8 — ΛCDM theoretical Hubble diagram

$$\mu_{\rm ΛCDM}(z) = 5\log_{10}\!\left(d_L / 10\,{\rm pc}\right)
= 5\log_{10}\!\left(d_L / {\rm Mpc}\right) + 25$$

where $d_L(z)=(1+z)\,d_C(z)$ and $d_C(z)=\frac{c}{H_0}\int_0^z \frac{dz'}{E(z')}$
with $E(z)=\sqrt{\Omega_m(1+z)^3+\Omega_\Lambda}$ for a flat ΛCDM universe.

In [ ]:
C_LIGHT = 2.998e5  # km/s


def E_func(z, Om=OmegaM, Ol=OmegaL):
    """Dimensionless Hubble parameter E(z) = H(z)/H0 for flat ΛCDM."""
    return np.sqrt(Om * (1.0 + z) ** 3 + Ol)


def distance_modulus_lcdm(z_arr, H0=H0, Om=OmegaM, Ol=OmegaL):
    """ΛCDM theoretical distance modulus μ(z) for an array of redshifts."""
    mu = np.zeros(len(z_arr), dtype=float)
    for i, z in enumerate(z_arr):
        if z <= 0:
            mu[i] = np.nan
            continue
        chi, _ = quad(lambda zp: 1.0 / E_func(zp, Om, Ol), 0.0, z)
        dc = (C_LIGHT / H0) * chi  # Mpc
        dl = dc * (1.0 + z)  # Mpc
        mu[i] = 5.0 * np.log10(dl) + 25.0
    return mu


def distance_modulus_empty(z_arr, H0=H0):
    """Empty-universe distance modulus (Ωm=0, ΩΛ=0)."""
    return distance_modulus_lcdm(z_arr, H0=H0, Om=0.0, Ol=0.0)


z_test = 0.5
print(f"Sanity check: ΛCDM μ(z=0.5) = {distance_modulus_lcdm(np.array([z_test]))[0]:.3f}  (expected ≈42.4)")

## 9 — Hubble diagram: μ(z_phot) vs ΛCDM

**Colour** = χ²/dof of the SALT2 fit (via a diverging colormap centred at 1).
**Marker shape** = galaxy type (elliptical, spiral, irregular, galaxy_DR8, mangrove, unknown).

In [ ]:
# ── Galaxy type → marker shape mapping ───────────────────────────────────────
GTYPE_MARKERS = {
    "elliptical": "s",  # square
    "spiral": "D",  # diamond
    "irregular": "^",  # triangle up
    "galaxy_simbad": "P",  # plus-filled
    "galaxy_DR8": "o",  # circle
    "mangrove": "h",  # hexagon
    "star_DR8": "x",  # cross (likely contamination)
    "unknown": "*",  # star
}

# Default marker for any unlisted type
_DEFAULT_MARKER = "*"

# Apply quality cuts for the Hubble diagram
if len(results_df) > 0:
    hd = results_df[
        (results_df["chi2_red"] < CHI2_RED_MAX)
        & (results_df["x1"].abs() < X1_MAX)
        & (results_df["c"].abs() < C_MAX)
    ].copy()
    print(f"Hubble diagram: {len(hd)}/{len(results_df)} events after quality cuts.")
else:
    hd = pd.DataFrame()
    print("No successful fits — Hubble diagram will be empty.")

In [ ]:
# ── Theoretical curves ────────────────────────────────────────────────────────
z_theory = np.linspace(1e-3, Z_MAX * 1.05, 300)
mu_lcdm = distance_modulus_lcdm(z_theory)
mu_empty = distance_modulus_empty(z_theory)

# ── Colormap for χ²/dof ───────────────────────────────────────────────────────
# Use a log-scale normalisation: χ²/dof = 1 is the ideal fit
CMAP = plt.get_cmap("RdYlGn_r")  # green=good (chi2~1), red=bad (chi2>>1)
chi2_vmin = 0.0
chi2_vmax = min(CHI2_RED_MAX, hd["chi2_red"].max()) if len(hd) > 0 else 5.0
norm = mcolors.Normalize(vmin=chi2_vmin, vmax=chi2_vmax)

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
ax_hub = axes[0]
ax_res = axes[1]

# --- ΛCDM and empty universe reference lines ---
ax_hub.plot(
    z_theory,
    mu_lcdm,
    color="royalblue",
    lw=2.0,
    ls="-",
    label=rf"ΛCDM ($\Omega_m={OmegaM}$, $\Omega_\Lambda={OmegaL}$, $H_0={H0}$)",
)
ax_hub.plot(
    z_theory,
    mu_empty,
    color="grey",
    lw=1.5,
    ls="--",
    label=r"Empty universe ($\Omega_m=0$, $\Omega_\Lambda=0$)",
)

# --- Data points: one scatter call per galaxy type ---
legend_handles_gtype = []

if len(hd) > 0:
    gtypes_present = hd["galaxy_type"].unique()

    for gtype in gtypes_present:
        sub = hd[hd["galaxy_type"] == gtype]
        marker = GTYPE_MARKERS.get(gtype, _DEFAULT_MARKER)
        colors_pt = [CMAP(norm(v)) for v in sub["chi2_red"].values]

        # Draw each point individually to use per-point colours
        for _, pt in sub.iterrows():
            ax_hub.scatter(
                pt["z_best"],
                pt["mu_tripp"],
                color=CMAP(norm(pt["chi2_red"])),
                marker=marker,
                s=80,
                edgecolors="k",
                linewidths=0.4,
                zorder=5,
            )
        # Add one entry to the galaxy-type legend (grey = representative colour)
        legend_handles_gtype.append(
            Line2D(
                [0],
                [0],
                marker=marker,
                color="w",
                markerfacecolor="dimgrey",
                markeredgecolor="k",
                markersize=8,
                label=gtype,
            )
        )

# --- Colorbar (χ²/dof) ---
sm = cm.ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax_hub, pad=0.02)
cbar.set_label(r"$\chi^2 / \mathrm{dof}$", fontsize=11)

# --- Axes and legend ---
ax_hub.set_ylabel(r"Distance modulus $\mu$", fontsize=12)
ax_hub.set_title(
    f"Hubble diagram — Fink/LSST  `{TAG}`\nSALT2 fit (z fixed to host photo-z)  |  N={len(hd)} events",
    fontsize=11,
)
ax_hub.grid(True, alpha=0.3)

# Two legends: ΛCDM lines + galaxy types
legend1 = ax_hub.legend(fontsize=9, loc="upper left")
legend2 = ax_hub.legend(
    handles=legend_handles_gtype, title="Galaxy type", title_fontsize=9, fontsize=8, loc="upper right"
)
ax_hub.add_artist(legend1)  # keep both legends

# --- Residuals panel ---
ax_res.axhline(0.0, color="royalblue", lw=1.5, ls="-")
ax_res.axhline(0.0, color="k", lw=0.8, ls="--")

if len(hd) > 0:
    mu_lcdm_obs = distance_modulus_lcdm(hd["z_best"].values)
    residuals = hd["mu_tripp"].values - mu_lcdm_obs

    for gtype in hd["galaxy_type"].unique():
        sub = hd[hd["galaxy_type"] == gtype]
        mask_idx = hd["galaxy_type"] == gtype
        res_sub = residuals[hd.index.get_indexer(sub.index)]
        marker = GTYPE_MARKERS.get(gtype, _DEFAULT_MARKER)
        for j, (_, pt) in enumerate(sub.iterrows()):
            mu_th = distance_modulus_lcdm(np.array([pt["z_best"]]))[0]
            res_val = pt["mu_tripp"] - mu_th
            ax_res.scatter(
                pt["z_best"],
                res_val,
                color=CMAP(norm(pt["chi2_red"])),
                marker=marker,
                s=60,
                edgecolors="k",
                linewidths=0.4,
                zorder=5,
            )

ax_res.set_xlabel("Photometric redshift $z_{\\rm phot}$", fontsize=12)
ax_res.set_ylabel(r"$\Delta\mu$ [mag]", fontsize=12)
ax_res.set_ylim(-3.0, 3.0)
ax_res.grid(True, alpha=0.3)

fig.tight_layout()
fig_path = FIGS_DIR / "hubble_diagram_zphot.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
print(f"Hubble diagram saved → {fig_path}")
plt.show()

## 10 — SALT2 parameter distributions (x₁, c, χ²/dof)

In [ ]:
if len(results_df) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # x1
    axes[0].hist(results_df["x1"], bins=15, color="steelblue", edgecolor="white")
    axes[0].axvline(0.0, color="k", ls="--", lw=1)
    axes[0].set_xlabel(r"SALT2 stretch $x_1$", fontsize=12)
    axes[0].set_ylabel("Count", fontsize=12)
    axes[0].set_title(r"$x_1$ distribution")

    # c
    axes[1].hist(results_df["c"], bins=15, color="tomato", edgecolor="white")
    axes[1].axvline(0.0, color="k", ls="--", lw=1)
    axes[1].set_xlabel(r"SALT2 colour $c$", fontsize=12)
    axes[1].set_ylabel("Count", fontsize=12)
    axes[1].set_title(r"$c$ distribution")

    # chi2_red
    axes[2].hist(results_df["chi2_red"], bins=15, color="seagreen", edgecolor="white")
    axes[2].axvline(1.0, color="k", ls="--", lw=1, label=r"$\chi^2$/dof = 1")
    axes[2].set_xlabel(r"$\chi^2 / \mathrm{dof}$", fontsize=12)
    axes[2].set_ylabel("Count", fontsize=12)
    axes[2].set_title(r"Reduced $\chi^2$")
    axes[2].legend()

    fig.suptitle(f"SALT2 parameter distributions — {TAG} (z fixed to photo-z)", fontsize=12)
    fig.tight_layout()
    fig.savefig(FIGS_DIR / "salt2_parameter_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Not enough successful fits to plot distributions.")

## 11 — Colour–stretch (x₁ vs c) plane coloured by galaxy type

In [ ]:
if len(results_df) >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))

    gtypes_present = results_df["galaxy_type"].unique()
    # Use a categorical colormap for galaxy types
    palette = plt.get_cmap("tab10")
    gtype_color_map = {
        g: palette(i / max(len(gtypes_present) - 1, 1)) for i, g in enumerate(sorted(gtypes_present))
    }

    legend_handles = []
    for gtype in sorted(gtypes_present):
        sub = results_df[results_df["galaxy_type"] == gtype]
        marker = GTYPE_MARKERS.get(gtype, _DEFAULT_MARKER)
        col = gtype_color_map[gtype]
        sc = ax.scatter(
            sub["c"],
            sub["x1"],
            c=[CMAP(norm(v)) for v in sub["chi2_red"]],
            marker=marker,
            s=90,
            edgecolors=col,
            linewidths=1.5,
            zorder=4,
        )
        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker=marker,
                color="w",
                markerfacecolor=col,
                markeredgecolor=col,
                markersize=9,
                label=f"{gtype} (N={len(sub)})",
            )
        )

    # Colorbar for chi2/dof
    sm2 = cm.ScalarMappable(cmap=CMAP, norm=norm)
    sm2.set_array([])
    cbar2 = fig.colorbar(sm2, ax=ax, pad=0.02)
    cbar2.set_label(r"$\chi^2/\mathrm{dof}$", fontsize=11)

    ax.axvline(0.0, color="grey", ls="--", lw=0.8)
    ax.axhline(0.0, color="grey", ls="--", lw=0.8)
    ax.set_xlabel(r"SALT2 colour $c$", fontsize=12)
    ax.set_ylabel(r"SALT2 stretch $x_1$", fontsize=12)
    ax.set_title(
        f"Colour–stretch plane — {TAG}\n(marker=galaxy type, edge=type colour, fill=χ²/dof)", fontsize=11
    )
    ax.legend(handles=legend_handles, title="Galaxy type", fontsize=9, title_fontsize=9, loc="best")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(FIGS_DIR / "salt2_colour_stretch_by_galtype.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Not enough data for colour–stretch plot.")

## 12 — Photometric redshift uncertainty propagation

We propagate the Legacy DR8 photo-z error `f:xm_legacydr8_e_zphot` into
an approximate uncertainty on the distance modulus:

$$\sigma_\mu \approx \frac{5}{\ln 10} \cdot \frac{1+z}{d_L} \cdot \frac{d(d_L)}{dz} \cdot \sigma_z
\approx \frac{5}{\ln 10} \cdot \frac{\sigma_z}{z}$$

(rough estimate for moderate z; more precisely computed numerically below).

In [ ]:
def dmu_dz(z, H0=H0, Om=OmegaM, Ol=OmegaL):
    """Numerical derivative dμ/dz at redshift z (finite difference)."""
    dz = 1e-4
    mu_p = distance_modulus_lcdm(np.array([z + dz]), H0, Om, Ol)[0]
    mu_m = distance_modulus_lcdm(np.array([z - dz]), H0, Om, Ol)[0]
    return (mu_p - mu_m) / (2 * dz)


if len(results_df) > 0 and results_df["z_phot_err"].notna().any():
    df_err = results_df[
        results_df["z_phot_err"].notna() & (results_df["z_source"] == "legacydr8_phot")
    ].copy()
    if len(df_err) > 0:
        df_err["sigma_mu_z"] = df_err.apply(lambda r: abs(dmu_dz(r["z_best"])) * r["z_phot_err"], axis=1)
        print("Photo-z uncertainty → μ uncertainty (σ_μ from σ_z):")
        print(df_err[["tns_name", "z_best", "z_phot_err", "mu_tripp", "sigma_mu_z"]].to_string(index=False))
    else:
        print("No objects with both z_phot_err and z_source=legacydr8_phot.")
else:
    print("No z_phot_err column or no valid values — skipping uncertainty propagation.")

## 13 — Hubble diagram with photo-z error bars

In [ ]:
YMIN = 35.0
YMAX = 48.0

if len(hd) == 0:
    print("No data for Hubble diagram with error bars.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
    ax_hub = axes[0]
    ax_res = axes[1]

    ax_hub.plot(
        z_theory, mu_lcdm, color="royalblue", lw=2.0, ls="-", label=rf"ΛCDM ($\Omega_m={OmegaM}$, $H_0={H0}$)"
    )
    ax_hub.plot(z_theory, mu_empty, color="grey", lw=1.5, ls="--", label=r"Empty universe")

    legend_handles_gtype2 = []
    gtypes_hd = hd["galaxy_type"].unique()

    for gtype in gtypes_hd:
        sub = hd[hd["galaxy_type"] == gtype]
        marker = GTYPE_MARKERS.get(gtype, _DEFAULT_MARKER)

        for _, pt in sub.iterrows():
            color_pt = CMAP(norm(pt["chi2_red"]))
            # x-error bar from photo-z uncertainty
            xerr_val = pt["z_phot_err"] if np.isfinite(pt.get("z_phot_err", np.nan)) else 0.0
            # y-error: propagate z uncertainty into mu
            if np.isfinite(xerr_val) and xerr_val > 0:
                yerr_val = abs(dmu_dz(pt["z_best"])) * xerr_val
            else:
                yerr_val = 0.0

            ax_hub.errorbar(
                pt["z_best"],
                pt["mu_tripp"],
                xerr=xerr_val if xerr_val > 0 else None,
                yerr=yerr_val if yerr_val > 0 else None,
                color=color_pt,
                fmt=marker,
                markersize=8,
                capsize=3,
                lw=0.8,
                zorder=5,
                markeredgecolor="k",
                markeredgewidth=0.4,
            )

        legend_handles_gtype2.append(
            Line2D(
                [0],
                [0],
                marker=marker,
                color="w",
                markerfacecolor="dimgrey",
                markeredgecolor="k",
                markersize=8,
                label=gtype,
            )
        )

    sm3 = cm.ScalarMappable(cmap=CMAP, norm=norm)
    sm3.set_array([])
    cbar3 = fig.colorbar(sm3, ax=ax_hub, pad=0.02)
    cbar3.set_label(r"$\chi^2/\mathrm{dof}$", fontsize=11)

    ax_hub.set_ylabel(r"Distance modulus $\mu$", fontsize=12)
    ax_hub.set_title(
        f"Hubble diagram — Fink/LSST  `{TAG}`\nSALT2 + photo-z (error bars from σ_z)  |  N={len(hd)}",
        fontsize=11,
    )
    ax_hub.grid(True, alpha=0.3)
    legend1b = ax_hub.legend(fontsize=9, loc="upper left")
    legend2b = ax_hub.legend(
        handles=legend_handles_gtype2, title="Galaxy type", title_fontsize=9, fontsize=8, loc="upper right"
    )
    ax_hub.add_artist(legend1b)
    ax_hub.set_xlim(0.0, Z_MAX)
    ax_hub.set_ylim(YMIN, YMAX)

    # Residuals
    ax_res.axhline(0.0, color="royalblue", lw=1.5, ls="-")
    ax_res.axhline(0.0, color="k", lw=0.8, ls="--")
    for _, pt in hd.iterrows():
        mu_th = distance_modulus_lcdm(np.array([pt["z_best"]]))[0]
        res_val = pt["mu_tripp"] - mu_th
        marker = GTYPE_MARKERS.get(pt["galaxy_type"], _DEFAULT_MARKER)
        color_pt = CMAP(norm(pt["chi2_red"]))
        xerr_val = pt.get("z_phot_err", np.nan)
        yerr_val = abs(dmu_dz(pt["z_best"])) * xerr_val if np.isfinite(xerr_val) and xerr_val > 0 else 0.0
        ax_res.errorbar(
            pt["z_best"],
            res_val,
            xerr=xerr_val if np.isfinite(xerr_val) and xerr_val > 0 else None,
            yerr=yerr_val if yerr_val > 0 else None,
            color=color_pt,
            fmt=marker,
            markersize=6,
            capsize=2,
            lw=0.7,
            zorder=5,
            markeredgecolor="k",
            markeredgewidth=0.3,
        )

    ax_res.set_xlabel(r"Photometric redshift $z_{\rm phot}$", fontsize=12)
    ax_res.set_ylabel(r"$\Delta\mu$ [mag]", fontsize=12)
    ax_res.set_ylim(-3.0, 3.0)
    ax_res.grid(True, alpha=0.3)

    fig.tight_layout()
    fig_path2 = FIGS_DIR / "hubble_diagram_zphot_errorbars.png"
    fig.savefig(fig_path2, dpi=150, bbox_inches="tight")
    print(f"Hubble diagram (with error bars) saved → {fig_path2}")
    plt.show()

## 14 — Summary table

In [ ]:
if len(results_df) > 0:
    display_cols = [
        "tns_name",
        "tns_type",
        "galaxy_type",
        "z_source",
        "z_best",
        "z_phot_err",
        "x1",
        "c",
        "mB",
        "mu_tripp",
        "chi2_red",
        "ndof",
    ]
    avail = [c for c in display_cols if c in results_df.columns]
    print(results_df[avail].to_string(index=False, float_format="{:.4f}".format))
else:
    print("No successful fits.")

## 15 — Notes and caveats

| Topic | Details |
|-------|---------|
| **Redshift prior** | Photo-z from Legacy Survey DR8 (`f:xm_legacydr8_zphot`) is used as a fixed prior. Its typical scatter is σ_z/(1+z) ≈ 0.02–0.06 for galaxies at z < 0.5. At higher z or for faint hosts, photo-z errors can be larger. |
| **Systematic bias** | Fixing z to an incorrect photo-z will bias x₀ (hence mB and μ). The Hubble residuals Δμ absorb this bias. |
| **Galaxy type** | The classification is coarse: it relies on SIMBAD otype strings and Legacy DR8 star/galaxy separator. No morphological fitting is performed. |
| **χ²/dof** | Values >> 1 indicate either poor LC coverage, host contamination, or a non-SNIa event. A cut `chi2_red < 10` is applied in the Hubble diagram. |
| **Tripp calibration** | α=0.14, β=3.14, M_B=−19.3 (Betoule+ 2014). These are not re-fitted; a full cosmological analysis would fit them simultaneously with the cosmological parameters. |
| **Comparison with notebook 03** | Notebook 03 uses the TNS *spectroscopic* z (more reliable). This notebook uses the near-galaxy *photometric* z, enabling a larger sample at the cost of higher redshift uncertainty. |
